# Introduction

Libraries:

In [1]:
import os
from pathlib import Path

# DATA WRANGLING, STATISTICS AND VISUALIZATION
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import randint, uniform, loguniform

# MODEL SELECTION AND OPTIMIZATION
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV

# DATA PREPROCESSING
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# CLASSIFICATION MODELS
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# EVALUATION METRICS
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, roc_auc_score

Assess `accuracy`, `precision`, `recall`, `f1-score`:

In [2]:
# Evaluation metrics considered
metrics = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

# Number of models trained per search
j = 1000

# Data

In [5]:
# Are we working on google colab?
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    
    # If so, look for the files on the drive location
    BASE_DIR = Path('/content/drive/MyDrive/TFM/data') 
else:
    
    # If the session is running local
    BASE_DIR = Path('../data/') 

# Ahora, en todo el código usas BASE_DIR para cargar tus datos
ruta_procesados = BASE_DIR / 'clean' / 'clinical_data_selected_features.parquet'

In [6]:
ruta_procesados

WindowsPath('../data/clean/clinical_data_selected_features.parquet')

## Preprocessing

Separate features and target variable:

In [ ]:
# Features
X = df.drop("cardiac_event", axis=1)

# Target class manually encoded
y = df["cardiac_event"].map({"no":0, "yes":1})

Divide data set into train, validation and test set:

In [ ]:
# Divide into train (and validation) and test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y
    )

Divide columns according to their data type:

In [ ]:
# Numeric features names
numeric_features = X.select_dtypes(include=[np.number]).columns

# Categoric features names
categorical_features = X.select_dtypes(include="category").columns

Preprocessing pipeline:

In [ ]:
# Numeric data
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Categoric data
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Random forest

Ensemble the model:

In [ ]:
# Full pipeline
pipe_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Parameter grid
params_distributions_rf = {
    'classifier__n_estimators': randint(50, 200),
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': randint(2, 11)
}

# Random search
random_search_rf = RandomizedSearchCV(
    estimator=pipe_rf,
    param_distributions=params_distributions_rf,
    n_iter=j,
    cv=5,
    scoring=metrics,
    refit = 'roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

Train and optimize the model:

In [ ]:
random_search_rf.fit(X_train, y_train)

Collect the information of every model trained during the optimization:

In [ ]:
# Pass the results into a data frame
results_rf = pd.DataFrame(random_search_rf.cv_results_)

# Select relevant colums
relevant_cols = [
    'params',
    'mean_test_accuracy',
    'std_test_accuracy',
    'mean_test_precision',
    'std_test_precision',
    'mean_test_recall',
    'std_test_recall',
    'mean_test_f1',
    'std_test_f1',
    'mean_test_roc_auc',
    'std_test_roc_auc',
    'rank_test_roc_auc'
]

results_summary_rf = results_rf[relevant_cols].sort_values(by='rank_test_roc_auc')

# Show the best 5 models
results_summary_rf.head()

# Support Vector Machine


Ensemble the model:

In [ ]:
# Full pipeline
pipe_svm = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(random_state=42))
])

# Parameter grid
params_distributions_svm = {
    'classifier__C': loguniform(1e-3, 1e3),
    'classifier__kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'classifier__gamma': ['scale', 'auto'] + list(loguniform(1e-4, 1e1).rvs(10)), 
    'classifier__degree': randint(2, 5),
    'classifier__class_weight': [None, 'balanced']
}

# Random search
random_search_svm = RandomizedSearchCV(
    estimator=pipe_svm,
    param_distributions=params_distributions_svm,
    n_iter=j,
    cv=5,
    scoring=metrics,
    refit = 'roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

Train and optimize the model:

In [ ]:
random_search_svm.fit(X_train, y_train)

Collect the information of every model trained during the optimization:

In [ ]:
# Pass the results into a data frame
results_svm = pd.DataFrame(random_search_svm.cv_results_)

# Select relevant colums
results_summary_svm = results_svm[relevant_cols].sort_values(by='rank_test_roc_auc')

# Show the best 5 models
results_summary_svm.head()

# K-Nearest Neighbors

Ensemble the model:

In [ ]:
# Full pipeline
pipe_knn = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier())
])

# Parameter grid
params_distributions_knn = {
    'classifier__n_neighbors': randint(3, 50),
    'classifier__weights': ['uniform', 'distance'],
    'classifier__p': [1, 2],
    'classifier__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}

# Random search
random_search_knn = RandomizedSearchCV(
    estimator=pipe_knn,
    param_distributions=params_distributions_knn,
    n_iter=j,
    cv=5,
    scoring=metrics,
    refit='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

Train and optimize the model:

In [ ]:
random_search_knn.fit(X_train, y_train)

Collect the information of every model trained during the optimization:

In [ ]:
# Pass the results into a data frame
results_knn = pd.DataFrame(random_search_knn.cv_results_)

# Select relevant colums
results_summary_knn = results_knn[relevant_cols].sort_values(by='rank_test_roc_auc')

# Show the best 5 models
results_summary_knn.head()